# Raw to Bronze

In [0]:
%run ./00_setup_and_logging

# Setup and logging

Using workspace.dav_project


## Bronze tables

ok: bronze_commits
ok: bronze_issues
ok: bronze_pulls


ok: bronze_quarantine


...

ok: silver_commit, silver_issue, silver_pull_request, silver_quarantine


## Log table

ok: pipeline_execution_logs


## Small helper functions

helpers ready: new_run_id, now_utc, log_run, get_merge_metrics


In [0]:
dbutils.widgets.text("repos",
    "pandas-dev/pandas,numpy/numpy,scikit-learn/scikit-learn,matplotlib/matplotlib,"
    "scipy/scipy,fastapi/fastapi,pallets/flask,pytest-dev/pytest",
    "Repos (comma separated)")
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental", "backfill"], "Load type")
dbutils.widgets.text("since_date", "", "Since date (YYYY-MM-DDTHH:MM:SSZ, blank = 24h ago)")
dbutils.widgets.text("until_date", "", "Until date (blank = no upper bound)")
dbutils.widgets.text("batch_id", "", "Batch ID (blank = auto)")
dbutils.widgets.text("github_token", "", "GitHub token (paste for this run only, clear before saving)")

REPOS = [r.strip() for r in dbutils.widgets.get("repos").split(",") if r.strip()]
LOAD_TYPE = dbutils.widgets.get("load_type")
SINCE_DATE = dbutils.widgets.get("since_date").strip() or None
UNTIL_DATE = dbutils.widgets.get("until_date").strip() or None
GITHUB_TOKEN = dbutils.widgets.get("github_token").strip() or None
BATCH_ID_OVERRIDE = dbutils.widgets.get("batch_id").strip() or None

print("repos:", REPOS)
print("load_type:", LOAD_TYPE, "| since:", SINCE_DATE, "| until:", UNTIL_DATE)

repos: ['pandas-dev/pandas', 'numpy/numpy', 'scikit-learn/scikit-learn', 'matplotlib/matplotlib', 'scipy/scipy', 'fastapi/fastapi', 'pallets/flask', 'pytest-dev/pytest']
load_type: incremental | since: None | until: None


In [0]:
import json
import time
import urllib.request
import urllib.error
import urllib.parse
from datetime import datetime, timedelta, timezone
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

if SINCE_DATE is None and LOAD_TYPE == "incremental":
    SINCE_DATE = (now_utc() - timedelta(days=1)).strftime("%Y-%m-%dT%H:%M:%SZ")
    print("since_date was blank, using 24h ago:", SINCE_DATE)
if SINCE_DATE is None and LOAD_TYPE == "full":
    SINCE_DATE = (now_utc() - timedelta(days=365)).strftime("%Y-%m-%dT%H:%M:%SZ")
    print("since_date was blank, using 12 months ago:", SINCE_DATE)

RUN_ID = new_run_id()
BATCH_ID = BATCH_ID_OVERRIDE or f"{LOAD_TYPE}_{(SINCE_DATE or 'all').replace(':','').replace('-','')}"
print("run_id:", RUN_ID, "| batch_id:", BATCH_ID)

since_date was blank, using 24h ago: 2026-10-10T10:42:03Z
run_id: c9ed3a62-bb9d-4430-96fb-48b92c9bc359 | batch_id: incremental_20261010T104203Z


## Explicit Bronze schema

In [0]:
BRONZE_SCHEMA = StructType([
    StructField("record_key", StringType(), False),
    StructField("source_repo", StringType(), False),
    StructField("entity_type", StringType(), False),
    StructField("raw_json", StringType(), False),
    StructField("api_params", StringType(), True),
    StructField("load_type", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("load_timestamp", TimestampType(), False),
])

QUARANTINE_SCHEMA = StructType([
    StructField("source_repo", StringType(), True),
    StructField("entity_type", StringType(), True),
    StructField("raw_record", StringType(), True),
    StructField("reason", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("load_timestamp", TimestampType(), True),
])

## Fetch one repo, one entity, one time window

In [0]:
PER_PAGE = 100


def _headers():
    h = {"Accept": "application/vnd.github+json"}
    if GITHUB_TOKEN:
        h["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return h


def _get(url, params):
    full = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(full, headers=_headers())
    while True:
        try:
            with urllib.request.urlopen(req, timeout=30) as resp:
                return resp.status, json.loads(resp.read().decode("utf-8")), dict(resp.headers)
        except urllib.error.HTTPError as e:
            if e.code in (403, 429) and e.headers.get("X-RateLimit-Remaining") == "0":
                reset = int(e.headers.get("X-RateLimit-Reset", time.time() + 30))
                wait = max(reset - time.time(), 1)
                print(f"  rate limited, sleeping {int(wait)}s")
                time.sleep(wait + 1)
                continue
            raise


def fetch_window(repo, entity, since_date, until_date):
    """Returns (records, params_used). records is a list of (key, raw_dict, reason_if_bad)."""
    base = f"https://api.github.com/repos/{repo}"
    records = []

    if entity == "commits":
        params = {"per_page": PER_PAGE}
        if since_date:
            params["since"] = since_date
        if until_date:
            params["until"] = until_date
        page = 1
        while True:
            params["page"] = page
            _, batch, _ = _get(f"{base}/commits", params)
            if not batch:
                break
            for rec in batch:
                key = rec.get("sha")
                records.append((key, rec, None if key else "missing sha"))
            if len(batch) < PER_PAGE:
                break
            page += 1
        return records, params

    endpoint = "issues" if entity == "issues" else "pulls"
    params = {"per_page": PER_PAGE, "state": "all", "sort": "updated", "direction": "desc"}
    if entity == "issues" and since_date:
        params["since"] = since_date 
    page = 1
    while True:
        params["page"] = page
        _, batch, _ = _get(f"{base}/{endpoint}", params)
        if not batch:
            break
        reached_floor = False
        for rec in batch:
            updated_at = rec.get("updated_at")
            if since_date and updated_at and updated_at < since_date:
                reached_floor = True
                continue
            if until_date and updated_at and updated_at > until_date:
                continue
            key = str(rec.get("id")) if rec.get("id") is not None else None
            records.append((key, rec, None if key else "missing id"))
        if reached_floor or len(batch) < PER_PAGE:
            break
        page += 1
    return records, params

## Fetch & load all repos

In [0]:
def load_entity_to_bronze(entity):
    start = now_utc()
    good_rows, bad_rows = [], []
    params_by_repo = {}
    try:
        for repo in REPOS:
            try:
                records, params = fetch_window(repo, entity, SINCE_DATE, UNTIL_DATE)
                params_by_repo[repo] = params
            except Exception as e:
                print(f"  {repo}/{entity} failed to fetch: {e}")
                continue
            for key, rec, reason in records:
                if reason:
                    bad_rows.append((repo, entity, json.dumps(rec), reason, BATCH_ID, RUN_ID, now_utc()))
                else:
                    good_rows.append((
                        key, repo, entity, json.dumps(rec),
                        json.dumps(params_by_repo[repo]), LOAD_TYPE, BATCH_ID, RUN_ID, now_utc(),
                    ))
            print(f"  {repo}/{entity}: {len(records)} records in window")

        inserted = updated = 0
        if good_rows:
            df = spark.createDataFrame(good_rows, schema=BRONZE_SCHEMA)
            df.createOrReplaceTempView("bronze_batch_view")
            full_table = f"{CATALOG}.{SCHEMA}.bronze_{entity}"
            spark.sql(f"""
                MERGE INTO {full_table} AS t
                USING bronze_batch_view AS s
                ON t.record_key = s.record_key
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)
            m = get_merge_metrics(full_table)
            inserted, updated = m["inserted"], m["updated"]

        if bad_rows:
            qdf = spark.createDataFrame(bad_rows, schema=QUARANTINE_SCHEMA)
            qdf.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_quarantine")

        log_run(RUN_ID, "Raw-to-Bronze", entity,
                {"repos": REPOS, "load_type": LOAD_TYPE, "since": SINCE_DATE,
                 "until": UNTIL_DATE, "batch_id": BATCH_ID},
                start, now_utc(), "SUCCESS",
                rows_inserted=inserted, rows_updated=updated, rows_quarantined=len(bad_rows))
        return inserted, updated, len(bad_rows)

    except Exception as e:
        log_run(RUN_ID, "Raw-to-Bronze", entity,
                {"repos": REPOS, "load_type": LOAD_TYPE, "since": SINCE_DATE,
                 "until": UNTIL_DATE, "batch_id": BATCH_ID},
                start, now_utc(), "FAILED", error_message=str(e))
        print(f"FAILED: {entity}: {e}")
        return 0, 0, 0


results = {}
for entity in ["commits", "issues", "pulls"]:
    print(f"\n=== {entity} ===")
    results[entity] = load_entity_to_bronze(entity)

print("\nSummary (inserted, updated, quarantined):")
for entity, r in results.items():
    print(f"  {entity}: {r}")


=== commits ===
  pandas-dev/pandas/commits: 36 records in window
  numpy/numpy/commits: 0 records in window
  scikit-learn/scikit-learn/commits: 0 records in window
  matplotlib/matplotlib/commits: 1 records in window
  scipy/scipy/commits: 1 records in window
  fastapi/fastapi/commits: 0 records in window
  pallets/flask/commits: 0 records in window
  pytest-dev/pytest/commits: 2 records in window
[log] Raw-to-Bronze / commits -> SUCCESS (inserted=0, updated=40, quarantined=0)

=== issues ===
  pandas-dev/pandas/issues: 189 records in window
  numpy/numpy/issues: 15 records in window
  scikit-learn/scikit-learn/issues: 28 records in window
  matplotlib/matplotlib/issues: 15 records in window
  scipy/scipy/issues: 18 records in window
  fastapi/fastapi/issues: 1 records in window
  pallets/flask/issues: 0 records in window
  pytest-dev/pytest/issues: 43 records in window
[log] Raw-to-Bronze / issues -> SUCCESS (inserted=1, updated=308, quarantined=0)

=== pulls ===
  pandas-dev/panda

## Check the proof

In [0]:
display(spark.sql(f"""
    SELECT run_id, layer, entity, status, rows_inserted, rows_updated, rows_quarantined, start_time
    FROM {CATALOG}.{SCHEMA}.pipeline_execution_logs
    WHERE run_id = '{RUN_ID}'
    ORDER BY start_time
"""))

run_id,layer,entity,status,rows_inserted,rows_updated,rows_quarantined,start_time
c9ed3a62-bb9d-4430-96fb-48b92c9bc359,Raw-to-Bronze,commits,SUCCESS,0,40,0,2026-10-11T10:42:04.081Z
c9ed3a62-bb9d-4430-96fb-48b92c9bc359,Raw-to-Bronze,issues,SUCCESS,1,308,0,2026-10-11T10:42:14.310Z
c9ed3a62-bb9d-4430-96fb-48b92c9bc359,Raw-to-Bronze,pulls,SUCCESS,1,223,0,2026-10-11T10:42:27.583Z
